#Notebook 2: Data Preparation & Feature Engineering


The quality of a machine learning model depends heavily on the quality of the input data. This notebook focuses on preparing the dataset for predictive modeling by separating features and target variables, creating train-test splits, identifying numerical and categorical features, and building a reusable preprocessing pipeline.

The resulting pipeline ensures that all future models receive consistently transformed inputs while preventing data leakage during training and evaluation.

# Import Libraries and Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler



In [ ]:
df=pd.read_csv("Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [ ]:
x=df.drop(columns=["LoanID","Default"])
y=df["Default"]


## Feature and Target Separation

Separate the predictor variables from the target variable (Default).

The target variable represents whether a borrower eventually defaulted on a loan, while all remaining features are treated as potential predictors of credit risk.

In [ ]:
num_cols=x.select_dtypes(include=["int64","float64"]).columns.to_list()
cat_cols=x.select_dtypes(include=["object"]).columns.to_list()


print("Numerical columns:",num_cols)
print("Categorical columns: ",cat_cols)

Numerical columns: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']
Categorical columns:  ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


# Train-Test SPlit

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,
                                               y,
                                               test_size=0.2,
                                               random_state=42,
                                               stratify=y)



## Verify Target Distribution

Validate that stratification successfully preserved the class distribution across training and testing datasets.

Maintaining similar distributions reduces evaluation bias and improves the reliability of performance metrics.

In [ ]:
print("Training target distribution: ")
print(y_train.value_counts(normalize=True)*100)
print("Testing target distribution: ")
print(y_test.value_counts(normalize=True)*100)

Training target distribution: 
Default
0    88.387337
1    11.612663
Name: proportion, dtype: float64
Testing target distribution: 
Default
0    88.386528
1    11.613472
Name: proportion, dtype: float64


## Build Preprocessing Pipelines

Machine learning models require numerical input and are often sensitive to feature scales.

Separate preprocessing pipelines are created for:

- Numerical Features
  - Median imputation
  - Standardization

- Categorical Features
  - Most-frequent imputation
  - One-Hot Encoding

This modular design improves maintainability and enables consistent preprocessing across training, evaluation, and deployment.

In [ ]:
num_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

cat_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])


## Unified Transformation Pipeline

Combine the numerical and categorical preprocessing workflows into a single ColumnTransformer.

This ensures that every dataset passes through the exact same transformation process, reducing the risk of preprocessing inconsistencies.

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ("num",num_pipeline,num_cols),
    ("cat",cat_pipeline,cat_cols)
])

In [ ]:
x_train_processed=preprocessor.fit_transform(x_train)
x_test_processed=preprocessor.transform(x_test)

## Validate Processed Features

Compare the original and transformed datasets to verify that preprocessing was applied successfully.

Feature expansion is expected because categorical variables are converted into multiple one-hot encoded features.

In [ ]:
print("Original training shape: ",x_train.shape)
print("Processed training shape: ",x_train_processed.shape)
print("Original testing shape: ",x_test.shape)
print("Processed testing shape: ",x_test_processed.shape)

Original training shape:  (204277, 16)
Processed training shape:  (204277, 31)
Original testing shape:  (51070, 16)
Processed testing shape:  (51070, 31)


## Inspect Generated Features

Review the transformed feature names produced by the preprocessing pipeline.

This step is particularly useful for model interpretation and explainability in later notebooks.

In [ ]:
feature_names=preprocessor.get_feature_names_out()
feature_names

array(['num__Age', 'num__Income', 'num__LoanAmount', 'num__CreditScore',
       'num__MonthsEmployed', 'num__NumCreditLines', 'num__InterestRate',
       'num__LoanTerm', 'num__DTIRatio', "cat__Education_Bachelor's",
       'cat__Education_High School', "cat__Education_Master's",
       'cat__Education_PhD', 'cat__EmploymentType_Full-time',
       'cat__EmploymentType_Part-time',
       'cat__EmploymentType_Self-employed',
       'cat__EmploymentType_Unemployed', 'cat__MaritalStatus_Divorced',
       'cat__MaritalStatus_Married', 'cat__MaritalStatus_Single',
       'cat__HasMortgage_No', 'cat__HasMortgage_Yes',
       'cat__HasDependents_No', 'cat__HasDependents_Yes',
       'cat__LoanPurpose_Auto', 'cat__LoanPurpose_Business',
       'cat__LoanPurpose_Education', 'cat__LoanPurpose_Home',
       'cat__LoanPurpose_Other', 'cat__HasCoSigner_No',
       'cat__HasCoSigner_Yes'], dtype=object)

## Create Modeling DataFrames

Convert transformed feature matrices into DataFrames to improve readability and simplify downstream analysis.

In [ ]:
x_train_processed_df=pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index=x_train.index
    )

x_test_processed_df=pd.DataFrame(
    x_test_processed,
    columns=feature_names,
    index=x_test.index
    )

x_train_processed_df.head()

,num__Age,num__Income,num__LoanAmount,num__CreditScore,num__MonthsEmployed,num__NumCreditLines,num__InterestRate,num__LoanTerm,num__DTIRatio,cat__Education_Bachelor's,...,cat__HasMortgage_Yes,cat__HasDependents_No,cat__HasDependents_Yes,cat__LoanPurpose_Auto,cat__LoanPurpose_Business,cat__LoanPurpose_Education,cat__LoanPurpose_Home,cat__LoanPurpose_Other,cat__HasCoSigner_No,cat__HasCoSigner_Yes
15826,0.099486,-1.167307,1.698627,0.312274,-1.170591,-0.449335,-1.335928,1.414459,1.512942,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
147371,0.299620,1.319747,-0.864170,-0.505800,1.714171,0.445947,0.185568,0.707489,-0.045473,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
178180,0.232908,0.453497,-1.700954,0.903803,1.396847,0.445947,-1.201856,-0.706451,1.123338,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
126915,-0.100649,-1.191966,-1.432895,-1.449730,-1.661000,0.445947,0.723364,0.000519,1.123338,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
163930,-1.568299,0.434508,1.707671,-1.613345,0.416028,0.445947,0.898110,1.414459,-0.218630,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## Save Preprocessing Artifacts

Persist the preprocessing pipeline and transformed datasets for use in subsequent modeling notebooks.

In [ ]:
import joblib
x_train_processed_df.to_csv("x_train_processed.csv",index=False)
x_test_processed_df.to_csv("x_test_processed.csv",index=False)

y_train.to_csv("y_train.csv",index=False)
y_test.to_csv("y_test.csv",index=False)

joblib.dump(preprocessor,"preprocessor.pkl")

['preprocessor.pkl']

## Key Outcomes

- Features and target variables were separated successfully.
- Stratified train-test splitting preserved the original target distribution.
- Numerical and categorical variables were processed using dedicated pipelines.
- A unified preprocessing workflow was created using ColumnTransformer.
- The preprocessing pipeline was saved for reuse across training, evaluation, and deployment stages.
- The resulting datasets are now ready for model development in the next notebook.